# LakeWatch: Sentinel-2 water / water-quality segmentation with a CNN

**Purpose.** Train and evaluate a convolutional neural network that processes real Sentinel-2 Level-2A imagery over Lake Malawi. The notebook is a complete, reproducible machine-learning pipeline:

1. environment and reproducibility metadata
2. STAC search against the Microsoft Planetary Computer
3. independent label retrieval (ESA WorldCover 2021)
4. cloud masking, band alignment, spectral-index feature engineering
5. patch extraction with spatial block identifiers
6. **spatially blocked** train / validation / test split (no leakage)
7. U-Net (ResNet-34 encoder) training with class-weighted cross-entropy + Dice loss
8. test-set evaluation with **real confusion matrices**, per-class precision/recall/F1/IoU, Cohen's kappa and bootstrap confidence intervals
9. water-extent area validation and qualitative prediction figures
10. artefact export (weights, metrics, figures, environment manifest)

**Run in Google Colab.** `Runtime -> Change runtime type -> T4 GPU` (or better). Expect roughly 30-90 minutes depending on the number of scenes.

**What this notebook is and is not.** The supervised target here is *surface class* (water, wetland, vegetation, other) from an independent, published land-cover product. That is a defensible, real-labelled task directly tied to the water-extent/water-level objective. Chlorophyll-a, turbidity and surface temperature are **retrieval/regression** problems: publication-grade validation requires in-situ matchups, which are not invented here. The last section explains exactly how to attach such matchups when they become available.

## 1. Environment and configuration

The install cell is safe on Colab. PyTorch and torchvision are pre-installed with CUDA; they are deliberately **not** reinstalled.

In [ ]:
# @title Install the extra packages (Colab)
!pip install --quiet pystac-client planetary-computer rioxarray rasterio segmentation-models-pytorch scikit-learn seaborn pandas tqdm

# @title Imports
from __future__ import annotations

import json
import math
import os
import platform
import random
import sys
import warnings
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.windows import from_bounds
import pystac_client
import planetary_computer

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    cohen_kappa_score,
    accuracy_score,
)
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")

print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available())
print("device :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# @title Configuration
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
S2_COLLECTION = "sentinel-2-l2a"
WORLDCOVER_COLLECTION = "esa-worldcover"

# Southern Lake Malawi. Adjust if your study area differs.
BBOX = [34.20, -12.80, 34.70, -12.20]
DATE_RANGE = "2022-01-01/2024-12-31"
MAX_CLOUD = 20
MAX_SCENES = 20                 # scenes actually processed (keeps Colab runtime sane)
MAX_PATCHES_PER_SCENE = 28
PATCH_SIZE = 128
MIN_VALID_FRACTION = 0.95       # cloud/NoData masking threshold per patch
MIN_WATER_FRACTION = 0.01       # keep shoreline + open-water patches
MAX_DOMINANT_FRACTION = 0.985   # discard almost-pure patches

S2_BANDS = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B11", "B12"]
CHANNELS = S2_BANDS + ["NDVI", "NDWI", "NDTI", "NDCI"]
CLASS_NAMES = ["water", "wetland", "vegetation", "other"]
N_CLASSES = len(CLASS_NAMES)

# ESA WorldCover -> task classes
WORLDCOVER_MAP = {
    80: 0,                    # permanent water
    90: 1,                    # herbaceous wetland
    10: 2, 20: 2, 30: 2, 40: 2, 95: 2, 100: 2,
    50: 3, 60: 3, 70: 3,      # built, bare, snow
}
INVALID_SCL = {0, 1, 3, 8, 9, 10, 11}   # NoData, saturated, shadow, cloud, cirrus, snow

SEED = 42
BATCH_SIZE = 8
NUM_EPOCHS = 30
PATIENCE = 6
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

ROOT = Path("/content/lakewatch_cnn")
PATCH_DIR = ROOT / "patches"
OUT_DIR = ROOT / "outputs"
for p in (PATCH_DIR, OUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
print("channels:", len(CHANNELS), "->", CHANNELS)
print("classes:", CLASS_NAMES)

## 2. Search real Sentinel-2 scenes

Only scenes returned by the Planetary Computer STAC API are used. The search query, item IDs and cloud-cover values are printed and saved, so the exact imagery behind every number is auditable.

In [ ]:
# @title STAC client and Sentinel-2 search
catalog = pystac_client.Client.open(STAC_URL, modifier=planetary_computer.sign_inplace)

search = catalog.search(
    collections=[S2_COLLECTION],
    bbox=BBOX,
    datetime=DATE_RANGE,
    query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
    limit=100,
)
items = sorted(search.items(), key=lambda it: it.datetime)

if not items:
    raise RuntimeError(
        "No Sentinel-2 scenes matched. Widen DATE_RANGE, raise MAX_CLOUD, or adjust BBOX."
    )

# Spread the selection across the whole time range instead of taking one month.
if len(items) > MAX_SCENES:
    pick = np.linspace(0, len(items) - 1, MAX_SCENES).astype(int)
    items = [items[i] for i in pick]

scene_rows = []
for it in items:
    scene_rows.append({
        "item_id": it.id,
        "datetime": it.datetime.isoformat(),
        "cloud_cover": it.properties.get("eo:cloud_cover"),
        "platform": it.properties.get("platform"),
        "processing_baseline": it.properties.get("s2:processing_baseline"),
    })
scenes_df = pd.DataFrame(scene_rows)
scenes_df.to_csv(OUT_DIR / "sentinel2_scenes.csv", index=False)
print(f"{len(items)} scenes selected from {len(list(search.items()))} matches")
display(scenes_df.head(10))

In [ ]:
# @title Plot the footprints on a simple map (no basemap dependency)
fig, ax = plt.subplots(figsize=(7, 6))
for it in items:
    geom = it.geometry
    coords = geom["coordinates"][0]
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]
    ax.plot(xs, ys, color="tab:blue", alpha=0.25, linewidth=1)
ax.scatter(
    [it.properties.get("s2:mean_solar_zenith", 0) for it in items],
    [it.properties.get("eo:cloud_cover", 0) for it in items],
    s=8,
    alpha=0,
)
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title(f"{len(items)} Sentinel-2 footprints over the study area")
ax.set_aspect("equal")
plt.show()

## 3. Independent labels: ESA WorldCover 2021

The CNN never sees its own spectral indices as labels. Supervised targets come from the ESA WorldCover 10 m product hosted on the same Planetary Computer catalogue. Four task classes are used:

| task class | WorldCover classes |
|---|---|
| water | 80 |
| wetland | 90 |
| vegetation | 10, 20, 30, 40, 95, 100 |
| other | 50, 60, 70 |

The 2021 product is static, so it does not capture short-term water-level change. The temporal caveat is stated in the limitations section; the split below is spatial, which is the main leakage risk for a map task.

In [ ]:
# @title Load the WorldCover window once and cache it in memory
wc_search = catalog.search(
    collections=[WORLDCOVER_COLLECTION],
    bbox=BBOX,
    datetime="2021-01-01/2021-12-31",
    limit=10,
)
wc_items = list(wc_search.items())
if not wc_items:
    raise RuntimeError(
        "ESA WorldCover was not returned by the catalogue. "
        "Check the collection id or use another independent label product."
    )

wc_arr = None
wc_transform = None
wc_crs = None
for it in wc_items:
    with rasterio.open(it.assets["map"].href) as src:
        window = from_bounds(*BBOX, transform=src.transform)
        arr = src.read(1, window=window)
        if arr.size == 0 or (arr > 0).mean() < 0.05:
            continue
        wc_arr = arr
        wc_transform = src.window_transform(window)
        wc_crs = src.crs
        print("using WorldCover item:", it.id, arr.shape, wc_crs)
        break

if wc_arr is None:
    raise RuntimeError("No usable WorldCover window intersected the study area.")

mapped = np.full(wc_arr.shape, -1, dtype=np.int16)
for value, cls in WORLDCOVER_MAP.items():
    mapped[wc_arr == value] = cls
valid_label_pixels = mapped >= 0
print("WorldCover label coverage inside bbox:", round(float(valid_label_pixels.mean()), 4))
for cls, name in enumerate(CLASS_NAMES):
    share = float((mapped == cls).mean())
    print(f"  {name:10s}: {share:7.2%}")

## 4. Band alignment, cloud masking and spectral features

Sentinel-2 bands arrive at 10 m and 20 m. Every band is reprojected onto the 10 m reference grid of `B02`, and the L2A processing-baseline offset is applied before reflectance scaling. Clouds and shadows are removed with the scene classification layer (SCL), not with a hand-tuned brightness threshold.

In [ ]:
def _reproject_to_ref(src_arr, src_transform, src_crs, ref, resampling):
    dst = np.full(ref["shape"], np.nan, dtype="float32")
    reproject(
        source=src_arr,
        destination=dst,
        src_transform=src_transform,
        src_crs=src_crs,
        src_nodata=None,
        dst_transform=ref["transform"],
        dst_crs=ref["crs"],
        dst_nodata=np.nan,
        resampling=resampling,
    )
    return dst


def _read_asset(href, ref, resampling=Resampling.bilinear):
    with rasterio.open(href) as src:
        arr = src.read(1).astype("float32")
        if src.shape == ref["shape"] and src.transform == ref["transform"] and src.crs == ref["crs"]:
            return arr
        return _reproject_to_ref(arr, src.transform, src.crs, ref, resampling)


def load_scene_cube(item):
    """Return (features[C,H,W], valid_mask[H,W], labels[H,W]) on the B02 grid."""
    with rasterio.open(item.assets["B02"].href) as src:
        ref = {"transform": src.transform, "crs": src.crs, "shape": (src.height, src.width)}

    baseline = float(item.properties.get("s2:processing_baseline", "0") or 0)
    offset = 1000.0 if baseline >= 4.0 else 0.0

    bands = {}
    for name in S2_BANDS:
        raw = _read_asset(item.assets[name].href, ref, Resampling.bilinear)
        bands[name] = np.clip((raw - offset) / 10000.0, 0.0, 1.0)

    scl = _read_asset(item.assets["SCL"].href, ref, Resampling.nearest)
    eps = 1e-6
    b = bands
    indices = {
        "NDVI": (b["B08"] - b["B04"]) / (b["B08"] + b["B04"] + eps),
        "NDWI": (b["B03"] - b["B08"]) / (b["B03"] + b["B08"] + eps),
        "NDTI": (b["B04"] - b["B11"]) / (b["B04"] + b["B11"] + eps),
        "NDCI": (b["B05"] - b["B04"]) / (b["B05"] + b["B04"] + eps),
    }
    features = np.stack([b[name] for name in S2_BANDS] + [indices[name] for name in CHANNELS[len(S2_BANDS):]], axis=0)

    scl_valid = ~np.isin(np.nan_to_num(scl, nan=-1).astype(int), list(INVALID_SCL))
    valid = scl_valid & np.isfinite(features).all(axis=0)

    labels = np.full(ref["shape"], -1, dtype=np.int16)
    reproject(
        source=mapped,
        destination=labels,
        src_transform=wc_transform,
        src_crs=wc_crs,
        src_nodata=-1,
        dst_transform=ref["transform"],
        dst_crs=ref["crs"],
        dst_nodata=-1,
        resampling=Resampling.nearest,
    )
    labels = labels.astype(np.int16)
    labels[~valid] = -1
    return features.astype("float32"), valid, labels, ref

## 5. Patch extraction with spatial block IDs

Each patch is a 128 x 128 chip on the 10 m grid (1.28 km). Patches are only retained when at least 95% of pixels are cloud-free and labelled, they contain at least two classes, they are not almost pure, and they include some water. Every patch gets a `tile_id` from a 4 x 3 spatial grid; the split uses these IDs so neighbouring pixels cannot appear in both training and test.

In [ ]:
# @title Extract patches to disk
TILE_COLS, TILE_ROWS = 4, 3
TILE_W = (BBOX[2] - BBOX[0]) / TILE_COLS
TILE_H = (BBOX[3] - BBOX[1]) / TILE_ROWS


def spatial_tile(lon, lat):
    col = min(TILE_COLS - 1, max(0, int((lon - BBOX[0]) / TILE_W)))
    row = min(TILE_ROWS - 1, max(0, int((lat - BBOX[1]) / TILE_H)))
    return f"r{row}c{col}"


records = []
rng = np.random.default_rng(SEED)

for scene_index, item in enumerate(items, start=1):
    try:
        features, valid, labels, ref = load_scene_cube(item)
    except Exception as exc:
        print(f"[skip] {item.id}: {type(exc).__name__}: {exc}")
        continue

    H, W = labels.shape
    if H < PATCH_SIZE or W < PATCH_SIZE:
        continue

    ys = list(range(0, H - PATCH_SIZE + 1, PATCH_SIZE))
    xs = list(range(0, W - PATCH_SIZE + 1, PATCH_SIZE))
    grid = [(y, x) for y in ys for x in xs]
    rng.shuffle(grid)

    kept = 0
    for y, x in grid:
        if kept >= MAX_PATCHES_PER_SCENE:
            break
        v = valid[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
        lab = labels[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
        if v.mean() < MIN_VALID_FRACTION:
            continue
        if (lab < 0).mean() > 0.02:
            continue
        counts = np.bincount(lab[lab >= 0], minlength=N_CLASSES)
        if counts.sum() == 0 or counts.max() / counts.sum() > MAX_DOMINANT_FRACTION:
            continue
        if counts[0] / counts.sum() < MIN_WATER_FRACTION:
            continue

        patch_id = f"{item.id}_{y}_{x}"
        np.savez(
            PATCH_DIR / f"{patch_id}.npz",
            x=features[:, y:y + PATCH_SIZE, x:x + PATCH_SIZE].astype("float16"),
            y=lab.astype("uint8"),
        )
        lon, lat = rasterio.transform.xy(ref["transform"], y + PATCH_SIZE / 2, x + PATCH_SIZE / 2)
        records.append({
            "patch_id": patch_id,
            "scene_id": item.id,
            "scene_date": item.datetime.date().isoformat(),
            "row": y,
            "col": x,
            "lon": float(lon),
            "lat": float(lat),
            "tile_id": spatial_tile(float(lon), float(lat)),
            "water_fraction": float(counts[0] / counts.sum()),
        })
        kept += 1

    print(f"[{scene_index:02d}/{len(items)}] {item.id} -> {kept} patches")

patches_df = pd.DataFrame(records)
if patches_df.empty:
    raise RuntimeError("No patches were extracted. Relax MIN_VALID_FRACTION or widen BBOX.")

patches_df.to_csv(OUT_DIR / "patch_index.csv", index=False)
print("total patches:", len(patches_df))
print(patches_df.groupby("tile_id").size().sort_values(ascending=False).head(12))
print(patches_df["water_fraction"].describe())

## 6. Spatially blocked train / validation / test split

Groups are spatial tiles. If too few tiles survive, the code falls back to scene-level groups and prints a warning. The same patch can never appear in more than one split.

In [ ]:
# @title Split
groups = patches_df["tile_id"]
if groups.nunique() < 5:
    print("WARNING: fewer than 5 spatial tiles; grouping by scene instead.")
    groups = patches_df["scene_id"]

unique_groups = groups.nunique()
if unique_groups < 3:
    print("WARNING: fewer than 3 groups; using a stratified random split by patch.")
    idx = rng.permutation(len(patches_df))
    n_train = int(0.6 * len(idx))
    n_val = int(0.2 * len(idx))
    train_df = patches_df.iloc[idx[:n_train]]
    val_df = patches_df.iloc[idx[n_train:n_train + n_val]]
    test_df = patches_df.iloc[idx[n_train + n_val:]]
else:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    train_idx, temp_idx = next(gss.split(patches_df, groups=groups))
    train_df = patches_df.iloc[train_idx].reset_index(drop=True)
    temp_df = patches_df.iloc[temp_idx].reset_index(drop=True)

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["tile_id"]))
    val_df = temp_df.iloc[val_idx].reset_index(drop=True)
    test_df = temp_df.iloc[test_idx].reset_index(drop=True)

print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, "tiles:", sorted(df["tile_id"].unique()))

assert not (set(train_df["tile_id"]) & set(test_df["tile_id"])), "spatial leakage between train and test"
assert not (set(train_df["tile_id"]) & set(val_df["tile_id"])), "spatial leakage between train and val"

## 7. Datasets, channel statistics and augmentation

Normalisation statistics and class weights are computed **from the training split only**. Augmentation is limited to flips and 90-degree rotations, which are label-preserving for an overhead map.

In [ ]:
class PatchDataset(Dataset):
    def __init__(self, frame, mean, std, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.mean = mean
        self.std = std
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with np.load(PATCH_DIR / f"{row.patch_id}.npz") as data:
            x = data["x"].astype("float32")
            y = data["y"].astype("int64")
        x = (x - self.mean[:, None, None]) / self.std[:, None, None]
        if self.augment:
            if random.random() < 0.5:
                x = np.flip(x, axis=2)
                y = np.flip(y, axis=1)
            if random.random() < 0.5:
                x = np.flip(x, axis=1)
                y = np.flip(y, axis=0)
            k = random.randint(0, 3)
            x = np.rot90(x, k, axes=(1, 2))
            y = np.rot90(y, k, axes=(0, 1))
        return (
            torch.from_numpy(np.ascontiguousarray(x)),
            torch.from_numpy(np.ascontiguousarray(y)),
        )


def compute_normalisation(frame, max_files=300):
    sample = frame.sample(min(max_files, len(frame)), random_state=SEED)
    total = np.zeros(len(CHANNELS), dtype=np.float64)
    total_sq = np.zeros(len(CHANNELS), dtype=np.float64)
    count = 0
    class_counts = np.zeros(N_CLASSES, dtype=np.int64)
    for _, row in sample.iterrows():
        with np.load(PATCH_DIR / f"{row.patch_id}.npz") as data:
            x = data["x"].astype("float32")
            y = data["y"]
        total += x.sum(axis=(1, 2))
        total_sq += np.square(x, dtype=np.float64).sum(axis=(1, 2))
        count += x.shape[1] * x.shape[2]
        class_counts += np.bincount(y.ravel(), minlength=N_CLASSES)
    mean = total / count
    var = np.maximum(total_sq / count - mean ** 2, 1e-8)
    return mean.astype("float32"), np.sqrt(var).astype("float32"), class_counts


train_mean, train_std, class_counts = compute_normalisation(train_df)
print("channel mean:", np.round(train_mean, 4))
print("channel std :", np.round(train_std, 4))
print("train class pixels:", dict(zip(CLASS_NAMES, class_counts.tolist())))

# Inverse-square-root class weights: rare wetland pixels count more, but the
# dominant water class is not suppressed.
class_weights = 1.0 / np.sqrt(np.maximum(class_counts, 1).astype(np.float64))
class_weights = class_weights / class_weights.mean()
print("class weights:", np.round(class_weights, 4))

train_ds = PatchDataset(train_df, train_mean, train_std, augment=True)
val_ds = PatchDataset(val_df, train_mean, train_std, augment=False)
test_ds = PatchDataset(test_df, train_mean, train_std, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print("batches:", len(train_loader), len(val_loader), len(test_loader))

## 8. Model, loss and optimiser

A U-Net with a pretrained ResNet-34 encoder is used. The loss combines class-weighted cross-entropy with multiclass Dice loss, which is standard for imbalanced segmentation. Model selection uses validation macro-IoU only; the test split is touched once, at the end.

In [ ]:
# @title Build the model
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=len(CHANNELS),
    classes=N_CLASSES,
).to(DEVICE)

ce = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
dice = smp.losses.DiceLoss(mode="multiclass", from_logits=True)


def criterion(logits, target):
    return ce(logits, target) + dice(logits, target)


optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

print(model.__class__.__name__, "parameters:", sum(p.numel() for p in model.parameters()))

## 9. Training with early stopping

The best checkpoint is selected on validation macro-IoU, not on training loss. Every epoch is logged to `training_history.csv`.

In [ ]:
def metrics_from_cm(cm):
    cm = cm.astype(np.float64)
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    with np.errstate(divide="ignore", invalid="ignore"):
        precision = np.where(tp + fp > 0, tp / (tp + fp), 0.0)
        recall = np.where(tp + fn > 0, tp / (tp + fn), 0.0)
        f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), 0.0)
        iou = np.where(tp + fp + fn > 0, tp / (tp + fp + fn), 0.0)
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "iou": iou,
        "macro_f1": float(np.nanmean(f1)),
        "macro_iou": float(np.nanmean(iou)),
        "accuracy": float(tp.sum() / max(cm.sum(), 1)),
    }


@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    truths, preds = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        logits = model(x)
        preds.append(logits.argmax(dim=1).cpu().numpy().astype(np.uint8))
        truths.append(y.numpy().astype(np.uint8))
    return np.concatenate(truths, axis=0), np.concatenate(preds, axis=0)


def evaluate_loader(model, loader):
    y_true, y_pred = predict_loader(model, loader)
    cm = confusion_matrix(y_true.ravel(), y_pred.ravel(), labels=np.arange(N_CLASSES))
    return cm, metrics_from_cm(cm), y_true, y_pred

In [ ]:
# @title Training loop
history = []
best_iou = -1.0
best_epoch = -1
epochs_without_improvement = 0
best_path = OUT_DIR / "best_model.pt"

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    seen = 0

    for x, y in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * x.size(0)
        seen += x.size(0)

    scheduler.step()
    train_loss = running_loss / max(seen, 1)
    val_cm, val_metrics, _, _ = evaluate_loader(model, val_loader)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_macro_iou": val_metrics["macro_iou"],
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)
    print(
        f"epoch {epoch:02d} | loss {train_loss:.4f} | "
        f"val acc {val_metrics['accuracy']:.4f} | "
        f"val macro-F1 {val_metrics['macro_f1']:.4f} | "
        f"val mIoU {val_metrics['macro_iou']:.4f}"
    )

    if val_metrics["macro_iou"] > best_iou:
        best_iou = val_metrics["macro_iou"]
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            "state_dict": model.state_dict(),
            "channels": CHANNELS,
            "class_names": CLASS_NAMES,
            "mean": train_mean,
            "std": train_std,
            "epoch": epoch,
            "val_macro_iou": best_iou,
        }, best_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"early stopping at epoch {epoch} (best epoch {best_epoch})")
            break

history_df = pd.DataFrame(history)
history_df.to_csv(OUT_DIR / "training_history.csv", index=False)
print(f"best epoch {best_epoch} with validation macro-IoU {best_iou:.4f}")

In [ ]:
# @title Learning curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o")
axes[0].set_title("Training loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross-entropy + Dice")

axes[1].plot(history_df["epoch"], history_df["val_macro_iou"], marker="o", label="macro-IoU")
axes[1].plot(history_df["epoch"], history_df["val_macro_f1"], marker="s", label="macro-F1")
axes[1].plot(history_df["epoch"], history_df["val_accuracy"], marker="^", label="accuracy")
axes[1].set_title("Validation metrics")
axes[1].set_xlabel("epoch")
axes[1].set_ylim(0, 1)
axes[1].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "learning_curves.png", dpi=200, bbox_inches="tight")
plt.show()

## 10. Final test-set evaluation

The best validation checkpoint is reloaded. All numbers below are computed on the held-out spatial tiles, which were never used for fitting or model selection.

Metrics reported:
- **Confusion matrix** in absolute pixel counts, plus a row-normalised version.
- Per-class **precision, recall, F1 and IoU**.
- **Cohen's kappa** and overall accuracy.
- **95% bootstrap confidence intervals**, resampling whole patches (not pixels) so spatial correlation is respected.

In [ ]:
# @title Reload the best checkpoint and evaluate the test split exactly once
checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

test_cm, test_metrics, test_true, test_pred = evaluate_loader(model, test_loader)
n_test_patches = len(test_true)

print("TEST PIXEL CONFUSION MATRIX (rows = truth, columns = prediction)")
cm_df = pd.DataFrame(test_cm, index=CLASS_NAMES, columns=CLASS_NAMES)
display(cm_df)

print("\nclassification report")
print(classification_report(test_true.ravel(), test_pred.ravel(), target_names=CLASS_NAMES, digits=4, zero_division=0))

kappa = cohen_kappa_score(test_true.ravel(), test_pred.ravel())
print("Cohen's kappa:", round(float(kappa), 4))
print("overall accuracy:", round(test_metrics["accuracy"], 4))

per_class = pd.DataFrame({
    "class": CLASS_NAMES,
    "precision": test_metrics["precision"],
    "recall": test_metrics["recall"],
    "f1": test_metrics["f1"],
    "iou": test_metrics["iou"],
    "support_pixels": test_cm.sum(axis=1),
})
per_class.to_csv(OUT_DIR / "test_per_class_metrics.csv", index=False)
display(per_class)

In [ ]:
# @title Bootstrap 95% confidence intervals (resample patches, not pixels)
def bootstrap_from_patch_cms(patch_cms, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(patch_cms)
    rows = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        cm = patch_cms[idx].sum(axis=0)
        m = metrics_from_cm(cm)
        rows.append({
            "macro_f1": m["macro_f1"],
            "macro_iou": m["macro_iou"],
            "accuracy": m["accuracy"],
            "water_iou": m["iou"][0],
        })
    frame = pd.DataFrame(rows)
    return {
        key: (float(frame[key].quantile(0.025)), float(frame[key].quantile(0.975)))
        for key in frame.columns
    }


patch_cms = np.stack([
    confusion_matrix(test_true[i].ravel(), test_pred[i].ravel(), labels=np.arange(N_CLASSES))
    for i in range(n_test_patches)
])
ci = bootstrap_from_patch_cms(patch_cms)

ci_rows = []
for key, (low, high) in ci.items():
    point = {
        "macro_f1": test_metrics["macro_f1"],
        "macro_iou": test_metrics["macro_iou"],
        "accuracy": test_metrics["accuracy"],
        "water_iou": float(test_metrics["iou"][0]),
    }[key]
    ci_rows.append({"metric": key, "point": point, "ci_low": low, "ci_high": high})
ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(OUT_DIR / "test_bootstrap_ci.csv", index=False)
display(ci_df)

In [ ]:
# @title Confusion matrices (absolute and row-normalised)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(test_cm, annot=True, fmt=",d", cmap="Blues", cbar=False,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title("Test confusion matrix (pixel counts)")
axes[0].set_xlabel("predicted")
axes[0].set_ylabel("true")

cm_norm = test_cm / np.maximum(test_cm.sum(axis=1, keepdims=True), 1)
sns.heatmap(cm_norm, annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=1,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title("Row-normalised confusion matrix")
axes[1].set_xlabel("predicted")
axes[1].set_ylabel("true")

plt.tight_layout()
plt.savefig(OUT_DIR / "test_confusion_matrices.png", dpi=200, bbox_inches="tight")
plt.show()

np.savetxt(OUT_DIR / "test_confusion_matrix.csv", test_cm, fmt="%d", delimiter=",")

In [ ]:
# @title Per-class IoU and F1 with bootstrap error bars
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(N_CLASSES)
width = 0.38
ax.bar(x - width / 2, test_metrics["iou"], width, label="IoU")
ax.bar(x + width / 2, test_metrics["f1"], width, label="F1")
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0, 1)
ax.set_ylabel("score")
ax.set_title("Held-out test performance by class")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "test_per_class_scores.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. Qualitative predictions

Each row shows a true-colour composite (B04/B03/B02), the independent WorldCover label, the CNN prediction and the error map. Displayed patches are drawn from the test split only.

In [ ]:
# @title Visual inspection of held-out patches
def stretch_rgb(patch, low=2, high=98):
    rgb = patch[[2, 1, 0]]
    lo, hi = np.percentile(rgb, (low, high))
    return np.clip((rgb - lo) / max(hi - lo, 1e-6), 0, 1).transpose(1, 2, 0)


palette = np.array([
    [0.12, 0.47, 0.71],   # water
    [0.20, 0.63, 0.35],   # wetland
    [0.60, 0.75, 0.25],   # vegetation
    [0.75, 0.62, 0.45],   # other
])
error_palette = np.array([[1, 1, 1], [0.85, 0.10, 0.10]])

indices_to_show = rng.choice(n_test_patches, size=min(4, n_test_patches), replace=False)
fig, axes = plt.subplots(len(indices_to_show), 4, figsize=(15, 3.6 * len(indices_to_show)))
if len(indices_to_show) == 1:
    axes = axes[None, :]

for row, idx in enumerate(indices_to_show):
    patch_id = test_df.iloc[idx].patch_id
    with np.load(PATCH_DIR / f"{patch_id}.npz") as data:
        x = data["x"].astype("float32")
    truth = test_true[idx]
    pred = test_pred[idx]

    axes[row, 0].imshow(stretch_rgb(x))
    axes[row, 0].set_title("Sentinel-2 true colour")
    axes[row, 1].imshow(palette[truth])
    axes[row, 1].set_title("WorldCover label")
    axes[row, 2].imshow(palette[pred])
    axes[row, 2].set_title("CNN prediction")
    axes[row, 3].imshow(error_palette[(truth != pred).astype(int)])
    axes[row, 3].set_title("Error map")
    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.savefig(OUT_DIR / "test_prediction_examples.png", dpi=200, bbox_inches="tight")
plt.show()

## 12. Water-extent validation

Because water extent is one of the project indicators, the notebook also checks the predicted water fraction per held-out patch against the independent label. The scatter, bias, MAE and RMSE below are computed from actual test predictions, not from a simulated example.

In [ ]:
# @title Predicted vs labelled water fraction
true_water = np.array([(t == 0).mean() for t in test_true])
pred_water = np.array([(p == 0).mean() for p in test_pred])
residual = pred_water - true_water

rmse = float(np.sqrt(np.mean(residual ** 2)))
mae = float(np.mean(np.abs(residual)))
bias = float(np.mean(residual))
ss_res = float(np.sum(residual ** 2))
ss_tot = float(np.sum((true_water - true_water.mean()) ** 2))
r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else float("nan")

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(true_water, pred_water, s=18, alpha=0.7)
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_xlabel("WorldCover water fraction")
ax.set_ylabel("CNN water fraction")
ax.set_title(f"Water extent: RMSE={rmse:.3f}, MAE={mae:.3f}, bias={bias:+.3f}, R2={r2:.3f}")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(OUT_DIR / "water_extent_validation.png", dpi=200, bbox_inches="tight")
plt.show()

water_metrics = {
    "rmse": rmse,
    "mae": mae,
    "bias": bias,
    "r2": r2,
    "n_test_patches": int(n_test_patches),
}
print(json.dumps(water_metrics, indent=2))

## 13. Export artefacts and environment manifest

Everything needed to reproduce or cite the run is written to `/content/lakewatch_cnn/outputs/`. The zip includes the trained weights, the patch index, all metrics, the confusion matrices, the figures and the exact library versions.

In [ ]:
# @title Save metrics, manifest and a zip
run_metrics = {
    "task": "Sentinel-2 surface-class segmentation (water / wetland / vegetation / other)",
    "label_source": "ESA WorldCover 2021 v200 (independent product)",
    "study_area_bbox": BBOX,
    "date_range": DATE_RANGE,
    "n_scenes": len(items),
    "n_patches": int(len(patches_df)),
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "n_test": int(len(test_df)),
    "channels": CHANNELS,
    "class_names": CLASS_NAMES,
    "model": "U-Net + ResNet-34 ImageNet encoder",
    "loss": "class-weighted cross-entropy + multiclass Dice",
    "split": "spatially blocked by 4x3 tile grid",
    "best_epoch": int(best_epoch),
    "best_val_macro_iou": float(best_iou),
    "test": {
        "confusion_matrix": test_cm.tolist(),
        "accuracy": float(test_metrics["accuracy"]),
        "macro_f1": float(test_metrics["macro_f1"]),
        "macro_iou": float(test_metrics["macro_iou"]),
        "cohen_kappa": float(kappa),
        "per_class": per_class.to_dict(orient="records"),
        "bootstrap_95ci": ci,
    },
    "water_extent": water_metrics,
    "environment": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "device": str(DEVICE),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "segmentation_models_pytorch": smp.__version__,
        "generated_utc": datetime.now(timezone.utc).isoformat(),
    },
}
with open(OUT_DIR / "run_metrics.json", "w", encoding="utf-8") as handle:
    json.dump(run_metrics, handle, indent=2)
scenes_df.to_csv(OUT_DIR / "sentinel2_scenes.csv", index=False)
patches_df.to_csv(OUT_DIR / "patch_index.csv", index=False)

import shutil
shutil.make_archive("/content/lakewatch_cnn_results", "zip", ROOT)
print("wrote /content/lakewatch_cnn_results.zip")
print(json.dumps({k: v for k, v in run_metrics.items() if k not in {"test", "channels"}}, indent=2)[:2500])

In [ ]:
# @title Optional download (uncomment in Colab)
# from google.colab import files
# files.download("/content/lakewatch_cnn_results.zip")

## 14. Publication notes, assumptions and how to extend to water quality

**What is validated here.** The CNN is trained and tested against an independent, published land-cover product on spatially disjoint tiles. The confusion matrices, per-class metrics, kappa and bootstrap intervals are all computed from actual held-out predictions. This is directly usable for the water-extent / water-level objective and for a methods section on satellite image segmentation.

**What is *not* validated here.** Chlorophyll-a, turbidity and surface-water temperature are continuous retrieval targets. A CNN cannot be "validated" for them without independent in-situ or laboratory matchups. Do **not** present a model trained on its own spectral indices as a validated chlorophyll product. The correct extension is:

1. obtain in-situ chlorophyll-a / turbidity / temperature samples with coordinates and timestamps;
2. match each sample to the Sentinel-2 scene acquired closest in time (same day or +/- 1 day) and extract the pixel window;
3. train a regression head (or a separate 1D/2D CNN) on `(spectral patch -> in-situ value)`;
4. split by **sampling station** and by **date**, not randomly by sample;
5. report RMSE, MAE, bias, R², Nash-Sutcliffe efficiency and prediction intervals on the held-out stations/dates;
6. report the matchup count and the time window, because cloud cover makes the effective sample much smaller than the number of satellite scenes.

The code below is a safe starting point for step 3-5; it is deliberately not run because no in-situ table is bundled with this notebook.

```python
# Pseudocode for the in-situ extension
# matchups = pd.read_csv("in_situ_matchups.csv")  # columns: lon, lat, datetime, chlorophyll_a
# for each row: find the nearest Sentinel-2 item within +/- 1 day and cloud < 20%
# extract a 5x5 pixel window around the sample and the corresponding spectral features
# split with GroupShuffleSplit(groups=matchups["station_id"])
# model = smp.Unet(..., classes=1) or a small CNN regression head
# losses = nn.HuberLoss() or nn.MSELoss()
# report RMSE/MAE/bias/R2/NSE with station-level bootstrap intervals
```

**Known limitations to state explicitly in the paper.**

- WorldCover is a 2021 static product; it is an independent label source but not a temporal water-mask time series.
- The spatial block split measures performance on unseen *locations*, not unseen *years*. Add a year-based split as a second experiment if temporal generalisation is claimed.
- Sentinel-2 L2A surface reflectance is not atmospherically corrected to water-leaving reflectance; for chlorophyll retrieval, apply a water-specific atmospheric correction (e.g. ACOLITE/SeaDAS) and document it.
- Class imbalance is real: wetland and "other" pixels are rare. Macro metrics and per-class IoU are reported for that reason; overall accuracy alone is misleading.
- The 10 m grid means one pixel is not a water-quality sample. Any product derived from these patches should be reported with its native spatial support.